# In the name of God
### HW6
### Deep Q-Learning




### Deep Q-Learning (DQN)

Certainly! Here's a more detailed explanation of the key concepts and components in the context of **Deep Q-Learning (DQN)** applied to the **Lunar Lander Problem**:

---

### **Deep Q-Learning (DQN)**

**Deep Q-Learning** is an advanced reinforcement learning (RL) algorithm that combines two powerful techniques:
1. **Q-Learning:** A classic model-free RL algorithm that learns the optimal action-value function (Q-function), which provides the expected future reward for an agent that takes a specific action in a given state.
2. **Deep Learning:** The Q-function in traditional Q-learning is typically stored as a table, but this becomes impractical for large state spaces (such as images). Deep Q-Learning addresses this by using deep neural networks (DNNs) to approximate the Q-function, allowing for the handling of complex, high-dimensional environments.

#### Key components of **DQN**:
- **Experience Replay:**  
  In reinforcement learning, the agent learns from experiences (state, action, reward, next state). However, in sequential data, consecutive experiences are often highly correlated, which can lead to unstable learning. **Experience Replay** mitigates this by storing past experiences in a **replay buffer**. When training the agent, experiences are randomly sampled from this buffer, breaking temporal correlations and leading to more stable learning.
  
- **Target Networks:**  
  The Q-values in Q-learning are updated using the Bellman equation, which relies on the current estimate of Q-values. In DQN, this can cause instability because the Q-values are continuously updated. To address this, **Target Networks** are used. A separate network (target network) is periodically updated to match the weights of the main Q-network. This stabilizes training by preventing the Q-values from changing too quickly.

---

### **The Lunar Lander Problem**

The **Lunar Lander Problem** is a reinforcement learning task where the goal is to train an agent to safely land a lunar lander on the surface of the moon. The agent must make decisions based on the lander's current state, including:

- **State Variables** (features of the environment):  
  - **Position (x, y):** The lander’s location in the 2D environment.
  - **Velocity (x, y):** The lander’s speed in both x and y directions.
  - **Angle:** The orientation of the lander.
  - **Angular Velocity:** The rate of change of the lander’s angle.

- **Actions:**  
  The agent has four possible actions to choose from at each time step:
  - **Thrust left:** Apply force to the left side of the lander.
  - **Thrust right:** Apply force to the right side of the lander.
  - **Thrust up:** Apply force upwards to counter gravity.
  - **Do nothing:** No action taken.

- **Objective:**  
  The goal is to control the lander’s actions such that it lands safely on the moon’s surface, ideally in the designated landing pad area. The agent receives rewards or penalties based on its performance, such as:
  - **Positive rewards** for landing safely.
  - **Negative rewards** for crashing or going off the screen.

---

### **Overview of the Lunar Lander Problem with DQN**

- **Environment:**  
  The problem uses the `LunarLander-v2` environment from **OpenAI Gym**, a toolkit for developing and comparing reinforcement learning algorithms. The environment provides the agent with feedback (rewards, next state) based on the actions it takes.

- **Techniques Used:**
  - **Deep Q-Learning (DQN):** The main algorithm used to solve this problem. The agent learns the optimal policy by approximating the Q-function with a neural network.
  - **Experience Replay:** Past experiences are stored in a replay buffer and sampled randomly to break correlations in the data and improve learning.
  - **Target Networks:** A separate, stable target network is used to calculate the target Q-values, which helps stabilize the learning process.

---

### **Instructions**

1. **Follow the code instructions:**  
   The notebook will guide you through implementing the components of the DQN algorithm step by step. For example, you'll define the experience replay buffer, implement the Q-network, and apply the target network. The code will contain `#####TO DO#####` placeholders where you need to insert your code.

2. **Experiment with hyperparameters:**  
   Hyperparameters like learning rate, discount factor (gamma), and the size of the replay buffer can significantly impact the agent’s learning performance. By experimenting with different values, you can see how they affect the training process and performance.

3. **Train and play the game:**  
   After training the agent, you can visualize its performance. The agent will try to land the lunar lander on the moon’s surface based on the policy it has learned.

4. **Answer provided questions:**  
   The notebook may include questions or tasks to help reinforce your understanding of how the algorithm works and how to adjust the hyperparameters to improve the agent’s performance.

---

### **Prerequisites**

Ensure you have the necessary libraries installed. Typically, you'll need:
- **TensorFlow or PyTorch:** For building and training the deep neural network used in DQN.
- **NumPy:** For numerical operations, especially when handling experience replay and network inputs.
- **OpenAI Gym:** For the environment simulation (`LunarLander-v2`).
- **Matplotlib/Seaborn:** For plotting training progress and visualizing results.



In [1]:
!pip install --upgrade setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [8]:
!pip install pygame==2.0.1

!pip install gym[box2d]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 67.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Using cached box2d-py-2.3.5.tar.gz (374 kB)
  Preparing metadata (setup.py) ... done
  Using cached pygame-2.1.0.tar.gz (5.8 MB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-faile

# Imports

In [5]:
import numpy as np
import gym
import time
import torch
import torch.nn as nn
import torch.optim as optim
import os
import collections
import matplotlib.pyplot as plt
import collections


env = gym.make('LunarLander-v2')

DependencyNotInstalled: box2D is not installed, run `pip install gym[box2d]`

The code begins by importing necessary libraries: **NumPy** for numerical operations, **Gym** for creating and managing the reinforcement learning environment, **time** for tracking the duration of episodes, and **PyTorch** for building and training deep neural networks. Specifically, `torch.nn` is used for defining neural network layers, and `torch.optim` is used for optimization techniques like gradient descent. The `collections` module is imported twice, but only one import is needed for data structures like deque. **Matplotlib** is also imported to visualize training results or performance metrics. The environment for the Lunar Lander problem is created using **Gym's** `gym.make('LunarLander-v2')`, which sets up the simulation where the agent learns to control the lander to safely land on the moon’s surface.

In [ ]:
class DQN(nn.Module):
    def __init__(self, in_features, n_actions):
        """
        Initialize the Deep Q-Network (DQN).

        Parameters:
        - in_features (int): Number of input features (dimension of the state).
        - n_actions (int): Number of possible actions in the environment.
        """
        super(DQN, self).__init__()

        self.layer1 = nn.Linear(in_features, 256)
        self.layer2 = nn.Linear(256, 128)
        self.layer3 = nn.Linear(128, 64)
        self.output_layer = nn.Linear(64, n_actions)

    def forward(self, state):
        """
        Define the forward pass of the network.

        Parameters:
        - state (torch.Tensor): The current state of the environment.

        Returns:
        - output (torch.Tensor): The Q-values for each action.
        """
        x = torch.relu(self.layer1(state))
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        output = self.output_layer(x)
        return output



The `DQN` class defines a neural network model for Deep Q-Learning, inheriting from `torch.nn.Module`. It consists of an initialization method (`__init__`) and a forward pass method (`forward`). In the initialization method, the network is structured with four layers: three hidden layers (`layer1`, `layer2`, `layer3`) with ReLU activation, followed by an output layer (`output_layer`) that predicts Q-values for each action. The `forward` method defines the data flow, where the input state is passed through the layers, with ReLU activations applied to the hidden layers, and the final output layer produces the Q-values. These Q-values represent the expected future rewards for each action in the given state.

In [ ]:
import collections
import numpy as np
import torch

class ExperienceBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def append(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size, device='cpu'):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[idx] for idx in indices])

        return (torch.tensor(states, dtype=torch.float32).to(device),
                torch.tensor(actions, dtype=torch.long).to(device),
                torch.tensor(rewards, dtype=torch.float32).to(device),
                torch.tensor(next_states, dtype=torch.float32).to(device),
                torch.tensor(dones, dtype=torch.float32).to(device))


The `ExperienceBuffer` class defines an experience replay buffer used in reinforcement learning algorithms like DQN. It has an initialization method (`__init__`), which sets up a deque to store experiences with a specified capacity. The `__len__` method returns the current size of the buffer. The `append` method adds a new experience (state, action, reward, next state, done) to the buffer. The `sample` method randomly samples a batch of experiences from the buffer, converts them into PyTorch tensors, and moves them to the specified device (CPU or GPU) for training. This class helps with stabilizing training by breaking temporal correlations and improving sample efficiency.

In [ ]:
class Agent():
    def __init__(self, env, buffer):
        self.env = env
        self.buffer = buffer
        self._reset()

    def _reset(self):
        self.state = self.env.reset()
        self.total_rewards = 0.0

    def step(self, net, eps, device="cpu"):
        done_reward = None

        if np.random.random() < eps:
            action = self.env.action_space.sample()
        else:
            state_tensor = torch.tensor([self.state], dtype=torch.float32).to(device)
            q_values = net(state_tensor)
            _, action_tensor = torch.max(q_values, dim=1)
            action = int(action_tensor.item())

        next_state, reward, done, _ = self.env.step(action)
        self.total_rewards += reward

        exp = (self.state, action, reward, next_state, done)
        self.buffer.append(exp)

        if done:
            done_reward = self.total_rewards
            self._reset()
        else:
            self.state = next_state

        return done_reward


The `Agent` class handles the agent’s interaction with the environment and stores experiences in the replay buffer.

- **`__init__(self, env, buffer)`** initializes the agent by setting up the environment and the replay buffer. It also calls the `_reset()` method to initialize the agent’s state and total rewards.

- **`_reset(self)`** resets the agent’s state to the initial state of the environment and sets the total rewards for the episode to zero.

- **`step(self, net, eps, device="cpu")`** handles one step of the agent’s interaction with the environment. The action is selected using an **epsilon-greedy** strategy: with probability `eps`, the agent explores by choosing a random action, and with probability `1 - eps`, it exploits its learned policy by selecting the action with the highest Q-value from the Q-network (`net`). After taking the action, the agent receives the next state, reward, and a done flag from the environment. The experience (state, action, reward, next state, done) is added to the replay buffer. If the episode is done, the total reward for the episode is returned, and the agent’s state is reset for the next episode. Otherwise, the state is updated to the next state.

In [ ]:
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_FINAL = 0.01
EPSILON_DECAY_OBS = 10**5
BATCH_SIZE = 32
MEAN_GOAL_REWARD = 250
REPLAY_BUFFER_SIZE = 10000
REPLAY_MIN_SIZE = 10000
LEARNING_RATE = 1e-4
SYNC_TARGET_OBS = 1000


- **`GAMMA = 0.99`**: Discount factor for future rewards, where a value of 0.99 indicates the agent values long-term rewards while still considering immediate rewards.
- **`EPSILON_START = 1.0`**: Initial exploration probability in the epsilon-greedy strategy, meaning the agent will initially explore (choose random actions) with 100% probability.
- **`EPSILON_FINAL = 0.01`**: Final exploration probability after training, at which point the agent mostly exploits its learned policy.
- **`EPSILON_DECAY_OBS = 10**5`**: Number of observations for epsilon decay, determining how many steps it takes for epsilon to decay from `EPSILON_START` to `EPSILON_FINAL`.
- **`BATCH_SIZE = 32`**: Size of the experience replay batch used to update the Q-network, with 32 experiences sampled per training step.
- **`MEAN_GOAL_REWARD = 250`**: The target mean reward considered to have solved the environment; the agent needs to consistently reach this reward to be considered successful.
- **`REPLAY_BUFFER_SIZE = 10000`**: The maximum capacity of the experience replay buffer; once this is reached, older experiences are discarded.
- **`REPLAY_MIN_SIZE = 10000`**: Minimum number of experiences in the replay buffer before training can begin, ensuring sufficient experience diversity.
- **`LEARNING_RATE = 1e-4`**: The learning rate for the neural network optimizer, determining the step size for updating the model's weights.
- **`SYNC_TARGET_OBS = 1000`**: Number of observations before synchronizing the target network with the online Q-network, stabilizing learning by keeping the target network's weights fixed for a while.



In [ ]:
import torch
import torch.nn as nn

def cal_loss(batch, net, tgt_net, device='cpu'):
    """
    TODO: Implement the loss calculation for Deep Q-Learning.

    Calculate the loss for Deep Q-Learning.

    Parameters:
    - batch (tuple): Batch of experiences (states, actions, rewards, dones, next_states).
    - net: The neural network representing the online Q-network.
    - tgt_net: The neural network representing the target Q-network.
    - device (str): Device for neural network computations (default is "cpu").

    Returns:
    - torch.Tensor: Loss value calculated using Mean Squared Error (MSE) loss.
    """

    states, actions, rewards, dones, next_states = batch

    states_v = torch.tensor(states).to(device)
    actions_v = torch.tensor(actions).to(device)
    rewards_v = torch.tensor(rewards).to(device)
    dones_v = torch.tensor(dones).to(device)
    next_states_v = torch.tensor(next_states).to(device)

    print(f"States Tensor Shape: {states_v.shape}, Type: {states_v.dtype}")
    print(f"Actions Tensor Shape: {actions_v.shape}, Type: {actions_v.dtype}")
    print(f"Rewards Tensor Shape: {rewards_v.shape}, Type: {rewards_v.dtype}")
    print(f"Dones Tensor Shape: {dones_v.shape}, Type: {dones_v.dtype}")
    print(f"Next States Tensor Shape: {next_states_v.shape}, Type: {next_states_v.dtype}")


    state_action_values = net(states_v).gather(1, actions_v.unsqueeze(-1)).squeeze(-1)
    next_state_values = tgt_net(next_states_v).max(1)[0]
    next_state_values[dones_v] = 0.0
    next_state_values = next_state_values.detach()
    expected_state_action_values = rewards_v + GAMMA * next_state_values
    loss = nn.MSELoss()(state_action_values, expected_state_action_values)

    return loss


The `cal_loss` function calculates the loss used for training a **Deep Q-Network (DQN)** by comparing the predicted Q-values to the expected Q-values using **Mean Squared Error (MSE)** loss.

The function first extracts the states, actions, rewards, dones, and next states from the provided batch of experiences. It then converts these values into tensors and moves them to the specified computation device (CPU or GPU). Afterward, the **state-action values** are predicted by passing the current states through the online Q-network (`net`). The `gather(1, actions_v.unsqueeze(-1))` operation selects the Q-values corresponding to the actions taken by the agent.

Next, the **next state values** are computed by passing the next states through the target Q-network (`tgt_net`). The maximum Q-value for each next state is selected using `max(1)[0]`, which corresponds to the best action the agent could take in the next state. If the episode has ended (i.e., `dones_v`), the next state values are set to zero for those states, as no future reward will be received.

The **expected Q-values** are then calculated using the Bellman equation:
$$
\text{Expected Q-values} = \text{Rewards} + \gamma \times \text{Next State Q-values}
$$
where `GAMMA` is the discount factor.

Finally, the **MSE loss** is calculated by comparing the predicted Q-values (`state_action_values`) to the expected Q-values (`expected_state_action_values`). This loss is returned to guide the optimization process in updating the Q-network.

# Learning Curves
 Plot learning curves showing key metrics (e.g., total rewards, loss) over the course of training. Analyze the trends and identify key points in the learning process.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

net = DQN(env.observation_space.shape[0], env.action_space.n).to(device)
tgt_net = DQN(env.observation_space.shape[0], env.action_space.n).to(device)

buffer = ExperienceBuffer(REPLAY_BUFFER_SIZE)

agent = Agent(env, buffer)

epsilon = EPSILON_START

optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)

total_rewards = []
losses = []

ts = time.time()
best_mean_reward = None
obs_id = 0

while True:
    obs_id += 1
    loss_t = None

    epsilon = max(EPSILON_FINAL, EPSILON_START - obs_id / EPSILON_DECAY_OBS)

    reward = agent.step(net, epsilon, device=device)

    if reward is not None:
        total_rewards.append(reward)
        game_time = time.time() - ts
        ts = time.time()
        mean_reward = np.mean(total_rewards[-100:])

        if best_mean_reward is None or best_mean_reward < mean_reward:
            torch.save(net.state_dict(), './lunar_lander-best.dat')
            best_mean_reward = mean_reward
            print("GAME : {}, TIME ECLAPSED : {}, EPSILON : {}, MEAN_REWARD : {}"
                  .format(obs_id, game_time, epsilon, mean_reward))
            if best_mean_reward - mean_reward > 10:
                print("Reward {} -> {} Model Saved".format(best_mean_reward, mean_reward))

        if mean_reward > MEAN_GOAL_REWARD:
            print("SOLVED in {} obs".format(obs_id))
            break

    if len(buffer) >= REPLAY_MIN_SIZE:
        batch = buffer.sample(BATCH_SIZE)
        optimizer.zero_grad()
        loss_t = cal_loss(batch, net, tgt_net, device=device)
        loss_t.backward()
        optimizer.step()

        if loss_t is not None:
            losses.append(loss_t.item())

    if obs_id % SYNC_TARGET_OBS == 0:
        tgt_net.load_state_dict(net.state_dict())

    if obs_id % 1000 == 0:
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.title("Total Rewards")
        plt.plot(total_rewards)
        plt.grid()

        plt.subplot(1, 2, 2)
        plt.title("Losses")
        plt.plot(losses)
        plt.grid()

        plt.show()



The code sets up and trains a **Deep Q-Network (DQN)** agent to solve the **Lunar Lander** problem using reinforcement learning. It starts by determining whether a GPU or CPU is available for training. Two DQN networks are created: one for the online Q-network (`net`) and one for the target Q-network (`tgt_net`). An experience buffer is also created to store the agent’s experiences during training. The agent interacts with the environment using the epsilon-greedy strategy, where it starts with full exploration (`epsilon = 1.0`) and gradually shifts towards exploitation (`epsilon` decays over time). The training loop involves the agent taking actions based on the current state, storing experiences in the buffer, and periodically training the Q-network using mini-batches of sampled experiences. The optimizer (Adam) is used to minimize the loss, which is calculated using the Mean Squared Error between predicted Q-values and expected Q-values. Every `SYNC_TARGET_OBS` steps, the target network is synchronized with the online network to stabilize the training process. The model is saved whenever a new best mean reward is achieved. The training loop continues until the agent’s mean reward surpasses a predefined goal, indicating that the problem has been solved. Additionally, plots of total rewards and losses are displayed every 1000 steps to monitor the training progress.

# Visual Comparison:

write a function to render and display the environment before and after training. What visual differences do you observe in the agent's behavior? Discuss it. Also, Upload the Videos with your notebook. You can use the following library for rendering and saving videos.

# Question:

Exploration (Epsilon-Greedy):

Discuss the significance of the exploration strategy, specifically the Epsilon-Greedy approach, in balancing exploration and exploitation during training.

In [ ]:
import imageio

def render_and_save_video(env, net, episodes=10, save_path="./render_video.mp4", device="cpu"):
    frames = []
    for ep in range(episodes):
        state = env.reset()
        total_reward = 0.0
        while True:
            state_v = torch.tensor([state], dtype=torch.float32).to(device)
            q_vals = net(state_v).detach()
            action = q_vals.max(1)[1].item()

            state, reward, done, _ = env.step(action)
            total_reward += reward
            frames.append(env.render(mode="rgb_array"))

            if done:
                break

    env.close()
    imageio.mimsave(save_path, frames, fps=20)


1. **Inputs**:
   - **`env`**: The environment in which the agent interacts (e.g., Lunar Lander).
   - **`net`**: The Q-network (trained DQN model) used to predict the best action.
   - **`episodes`**: The number of episodes for which the agent interacts with the environment (default is 10).
   - **`save_path`**: The path where the rendered video will be saved (default is `./render_video.mp4`).
   - **`device`**: The device on which to perform computations (either `"cpu"` or `"cuda"`).

2. **Process**:
   - **Frame Collection**: The function starts by initializing an empty list `frames` to store the frames (images) of the environment rendered during each step.
   - **Episode Loop**: The function runs through multiple episodes (specified by the `episodes` argument).
     - The environment is reset at the beginning of each episode to initialize the state.
     - **Action Selection**: For each state, the Q-network (`net`) is used to predict the Q-values. The action with the highest Q-value is selected (exploitation step of the epsilon-greedy policy).
     - The action is taken in the environment using `env.step(action)`, which returns the next state, reward, and a boolean flag indicating whether the episode is done.
     - **Rendering**: After each action, the environment’s state is rendered as an RGB image (`env.render(mode="rgb_array")`) and added to the `frames` list.
   - **End of Episode**: The loop continues until the episode is done, and the environment closes once all episodes are finished.

3. **Saving the Video**:
   - The frames are saved as a video using `imageio.mimsave(save_path, frames, fps=20)`, where `fps=20` specifies the frame rate for the video.

4. **Output**:
   - The function does not return anything, but it saves a video of the agent's interactions with the environment to the specified `save_path`.



In [ ]:
print("### BEFORE TRAINING ###")
render_and_save_video(env, net, device=device, save_path='./before.mp4')

print("### AFTER TRAINING ###")
render_and_save_video(env, net, device=device, save_path='./after.mp4')
